# Your first NEXUS model

Train and score a credit-risk classifier in nine cells.

In [1]:
import os
from pathlib import Path

# Load secrets from ../.env.local if FUNDAMENTAL_API_KEY isn't already in the env.
env_file = Path("../.env.local")
if env_file.exists() and "FUNDAMENTAL_API_KEY" not in os.environ:
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            value = value.strip().strip('"').strip("'")
            os.environ.setdefault(key.strip(), value)

from fundamental import Fundamental, NEXUSClassifier, set_client

# Reads FUNDAMENTAL_API_KEY from your environment.
set_client(Fundamental())

print("Connected.")

Connected.


## The data

We have 4,591 historical borrowers. Some defaulted on their loans, most did not. We are going to train a model that scores how likely a new borrower is to default.

In [2]:
import pandas as pd

train   = pd.read_csv("../dataset/credit_risk/borrowers_train.csv")
holdout = pd.read_csv("../dataset/credit_risk/borrowers_holdout.csv")

print(f"Train:   {train.shape[0]:,} rows, {train.shape[1]} columns")
print(f"Holdout: {holdout.shape[0]:,} rows, {holdout.shape[1]} columns")

train.head(3)

Train:   4,591 rows, 13 columns
Holdout: 1,149 rows, 13 columns


,borrower_id,first_name,last_name,age,gender,marital_status,distance_from_branch_miles,education_level,occupation_sector,num_previous_lenders,total_employment_years,account_open_date,default_flag
0,1,Linda,Smith,42,Female,Married,10,Bachelor's,Human Resources,0,21.0,2004-01-01,1
1,3,Jessica,Jones,36,Female,Divorced,8,Bachelor's,Medical,1,22.0,2004-01-01,0
2,4,Linda,Moore,32,Female,Married,19,Bachelor's,Life Sciences,1,11.0,2014-01-01,0


In [3]:
FEATURES = [
    "age",
    "distance_from_branch_miles",
    "num_previous_lenders",
    "total_employment_years",
]
TARGET = "default_flag"

X_train,   y_train   = train[FEATURES],   train[TARGET]
X_holdout, y_holdout = holdout[FEATURES], holdout[TARGET]

print(f"Default rate in training data: {y_train.mean():.1%}")

Default rate in training data: 19.1%


In [4]:
# Training takes a minute or two. Stay on this cell while it runs.
nexus = NEXUSClassifier(mode="quality")
nexus.fit(X_train, y_train)

print("Training complete.")

Training complete.


In [5]:
from sklearn.metrics import roc_auc_score

proba = nexus.predict_proba(X_holdout)[:, 1]
auc   = roc_auc_score(y_holdout, proba)

print(f"AUC on holdout: {auc:.3f}")

AUC on holdout: 0.799


In [6]:
# Pull the trained model's permanent ID off the classifier.
model_id = nexus.trained_model_id_
print(f"Trained model ID: {model_id}")

# Make a fresh classifier. No fit, no training. Load the model by ID.
nexus2 = NEXUSClassifier()
nexus2.load_model(model_id)

reload_auc = roc_auc_score(y_holdout, nexus2.predict_proba(X_holdout)[:, 1])
print(f"Reloaded AUC:     {reload_auc:.3f}")

Trained model ID: 1f3e7558-b29f-4b82-9000-87907d562e23


Reloaded AUC:     0.799


## Where to next

The full workshop covers categorical features, async patterns, resilient pipelines, diagnostics, and what production deployment looks like end to end.